# Tarea 1: Predicción de resultados del fútbol uruguayo

In [ ]:
# %pip install pandas numpy matplotlib scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\lucas\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

--- 
### 1. Carga del Dataset y Descripción de Atributos

In [4]:
DATASET_FILE = "./futbol_uruguayo.csv" 

dataset = pd.read_csv(DATASET_FILE)
dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national


#### Descripción de los atributos:

| Atributo | Descripción |
| :--- | :--- |
| **`home`** | Nombre del equipo local (no necesariamente único) |
| **`away`** | Nombre del equipo visitante (no necesariamente único) |
| **`date`** | Fecha del partido |
| **`gh`** | Goles del equipo local (incluyendo tiempo extra y penales) |
| **`ga`** | Goles del equipo visitante (incluyendo tiempo extra y penales) |
| **`full_time`** | "F"=el partido terminó en 90', "E"=tiempo extra, "P"=penales |
| **`competition`** | Nombre del país de la liga o nombre de la competición int. |
| **`home_ident`** | Identificador único del equipo local |
| **`away_ident`** | Identificador único del equipo visitante |
| **`home_country`** | País del equipo local |
| **`away_country`** | País del equipo visitante |
| **`home_code`** | Código de país del equipo local |
| **`away_code`** | Código de país del equipo visitante |
| **`home_continent`** | Continente del equipo local |
| **`away_continent`** | Continente del equipo visitante |
| **`continent`** | Continente de la competición |
| **`level`** | "national"= liga local, "international"= copa internacional |

--- 
### 2. Definición de la Variable Objetivo (`ganador`)

El objetivo del modelo es predecir el resultado final de un partido de fútbol, clasificándolo en una de tres categorías posibles: victoria local, victoria visitante o empate.

Dado que el dataset original no incluye directamente una columna de resultado, se deduce la variable objetivo **`ganador`** mediante la comparación de los goles anotados por el equipo local (`gh`) y el visitante (`ga`):

* Si $\text{gh} > \text{ga} \implies$ **`L`**
* Si $\text{ga} > \text{gh} \implies$ **`V`**
* Si $\text{gh} == \text{ga} \implies$ **`E`**

Concretado lo anterior, la información que proveen los atributos "gh" y "ga" ya se ve contemplada por la variable objetivo. Por lo tanto, su presencia en el dataset no agrega significancia al entrenamiento del modelo y se deben remover ambos atributos del dataset.

In [ ]:
dataset["ganador"] = np.select(
    [
        dataset["gh"] > dataset["ga"],
        dataset["gh"] < dataset["ga"],
        dataset["gh"] == dataset["ga"],
    ],
    [
        "L",
        "V",
        "E",
    ],
    default="sin_dato",
)

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level,ganador
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,V
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,empate
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,L


--- 
### 3. Selección de Atributos

En esta etapa del preprocesamiento se realiza una selección de los atributos del dataset con el objetivo de maximizar la capacidad de aprendizaje del modelo en la clasificación de las instancias. Para ello, se examina cada atributo de forma individual para evaluar si contribuye de manera significativa al entrenamiento o si, dadas sus características, puede ser removido sin perjudicar el desempeño.

* **Atributos constantes:** No contribuyen al aprendizaje del modelo debido a que mantienen el mismo valor para todas las instancias del dataset (varianza cero).
* **Atributos redundantes:** Representan valores equivalentes dentro del dataset, por lo que basta con conservar uno de ellos.

#### A. Detección de Atributos Constantes

In [6]:
# Detección de Atributos Constantes
constantes = dataset.nunique(dropna=False)[dataset.nunique(dropna=False) == 1]
print("Atributos constantes detectados:")
print(constantes)

Atributos constantes detectados:
competition       1
home_country      1
away_country      1
home_code         1
away_code         1
home_continent    1
away_continent    1
continent         1
level             1
dtype: int64


Los atributos constantes son los siguientes:
* `competition`, `home_country`, `away_country` (Todos refieren a Uruguay).
* `home_code`, `away_code` (Código constante `UY`).
* `home_continent`, `away_continent`, `continent` (Todos refieren a `South America`).
* `level` (Constante con el valor `national`).

Estos atributos no serán incluidos en los conjuntos de entrenamiento y evaluación.

#### B. Detección de Atributos Redundantes

In [7]:
# Verificar cuántos identificadores tiene cada nombre de equipo, y viceversa
home_name_to_id = dataset.groupby("home")["home_ident"].nunique(dropna=False)
home_id_to_name = dataset.groupby("home_ident")["home"].nunique(dropna=False)

away_name_to_id = dataset.groupby("away")["away_ident"].nunique(dropna=False)
away_id_to_name = dataset.groupby("away_ident")["away"].nunique(dropna=False)

print(f"Cada valor de home se corresponde a un solo valor de home_ident, y viceversa: ", (home_name_to_id == 1).all()  & (home_id_to_name == 1).all())
print(f"Cada valor de away se corresponde a un solo valor de away_ident, y viceversa: ", (away_name_to_id == 1).all()  & (away_id_to_name == 1).all())

Cada valor de home se corresponde a un solo valor de home_ident, y viceversa:  True
Cada valor de away se corresponde a un solo valor de away_ident, y viceversa:  True


Existe una correspondencia 1:1, mantener ambas variables introduciría información redundante. Elegimos quedarnos con `home` y `away` y se descartan sus identificadores (`home_ident` y `away_ident`).

#### C. Descomposición del atributo `date`

El atributo `date` se divide en los atributos `day`, `month` y `year` para representar sus componentes por separado. Esta transformación facilita que el modelo identifique patrones relacionados con el momento del año en que se disputó el partido, como diferencias entre meses, temporadas o períodos históricos. Además, evita tratar cada fecha completa como un valor independiente, lo que podría dificultar el aprendizaje. Una vez extraídos estos componentes, se debe remover el atributo `date` original para evitar mantener información redundante.

En el flujo final, este procedimiento será ejecutado dentro del `Pipeline`.

In [8]:
dataset["date"] = pd.to_datetime(dataset["date"])
dataset["day"] = dataset["date"].dt.day
dataset["month"] = dataset["date"].dt.month
dataset["year"] = dataset["date"].dt.year

dataset.head()

,home,away,date,gh,ga,full_time,competition,home_ident,away_ident,home_country,...,home_code,away_code,home_continent,away_continent,continent,level,ganador,day,month,year
0,Bella Vista,Defensor Sporting,1932-03-05,1.0,2.0,F,uruguay,Bella Vista (Uruguay),Defensor Sporting (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,V,5,3,1932
1,CA Penarol,River Plate,1932-03-05,1.0,1.0,F,uruguay,CA Penarol (Uruguay),River Plate (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,empate,5,3,1932
2,Montevideo Wanderers,Racing Club,1932-03-05,3.0,0.0,F,uruguay,Montevideo Wanderers (Uruguay),Racing Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
3,Central Espanol,Rampla Juniors Futbol Club,1932-03-05,1.0,0.0,F,uruguay,Central Espanol (Uruguay),Rampla Juniors Futbol Club (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932
4,Nacional,Institucion Atletica Sud America,1932-03-05,2.0,0.0,F,uruguay,Nacional (Uruguay),Institucion Atletica Sud America (Uruguay),uruguay,...,UY,UY,South America,South America,South America,national,L,5,3,1932


---
### 4. Análisis de Balance y Estratificación:

In [9]:
print(f"Cantidad total de instancias: {dataset.shape[0]}")
print(f"Cantidad total de atributos: {dataset.shape[1]}")

Cantidad total de instancias: 15207
Cantidad total de atributos: 21


In [10]:
# Cálculo de la dispersión de instancias respecto a las clases
dataset["ganador"].value_counts(normalize=True).mul(100).round(2)

ganador
L         44.35
V         28.20
empate    27.44
Name: proportion, dtype: float64

La distribución de la clase objetivo muestra las siguientes proporciones:
* **`L`**: **44.35%**
* **`V`**: **28.20%**
* **`E`**: **27.44%**

Aunque existe un predominio de las victorias locales, la distribución no presenta un desbalance crítico (como ocurriría en escenarios con clases minoritarias $< 5\%$), por lo que no se requiere la aplicación de técnicas de mitigación como *SMOTE* o *undersampling*. 

Sin embargo, para evitar que una división puramente aleatoria, se aplica un muestreo estratificado (`stratify=dataset_Y`). De esta manera, se garantiza que tanto el conjunto de entrenamiento como el de evaluación mantengan proporciónes similares a las originales del dataset para cada clase.

---
### 5. División del Conjunto de Datos (`train_test_split`)

Se debe dividir el conjunto dataset de la siguiente forma:

Conjunto de entrenamiento los partidos jugados hasta el año 2023 inclusive y como conjunto de evaluación los partidos jugados en 2024 y 2025.


In [ ]:
# Instancias con partidos jugados hasta el año 2023 inclusive
df_entrenamiento = dataset[dataset['year'] <= 2023]
X_train = df_entrenamiento.drop(columns=["ganador", "gh", "ga"])
Y_train = df_entrenamiento["ganador"]

df_evaluacion = dataset[dataset['year'] > 2023]
X_test = df_evaluacion.drop(columns=["ganador", "gh", "ga"])
Y_test = df_evaluacion["ganador"]

print(f"Dimensiones de X_train (Entrenamiento): {X_train.shape}")
print(f"Dimensiones de X_test (Evaluación):    {X_test.shape}")

Dimensiones de X_train (Entrenamiento): (14734, 18)
Dimensiones de X_test (Evaluación):    (473, 18)


,home,away,date,full_time,competition,home_ident,away_ident,home_country,away_country,home_code,away_code,home_continent,away_continent,continent,level,day,month,year
14734,Nacional,River Plate,2024-02-17,F,uruguay,Nacional (Uruguay),River Plate (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,17,2,2024
14735,CA Fenix,Danubio,2024-02-17,F,uruguay,CA Fenix (Uruguay),Danubio (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,17,2,2024
14736,Miramar Misiones,CA Progreso,2024-02-17,F,uruguay,Miramar Misiones (Uruguay),CA Progreso (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,17,2,2024
14737,Deportivo Maldonado,Boston River,2024-02-18,F,uruguay,Deportivo Maldonado (Uruguay),Boston River (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,18,2,2024
14738,CA Cerro,Montevideo Wanderers,2024-02-18,F,uruguay,CA Cerro (Uruguay),Montevideo Wanderers (Uruguay),uruguay,uruguay,UY,UY,South America,South America,South America,national,18,2,2024


In [ ]:
numeric_features = ['year', 'month', 'day']
categorical_features = ['home', 'away', 'full_time']

# Pipeline para los atributos numéricos
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer()) # Reemplaza los valores NaN por la media (mean) por defecto
])

# Pipeline para los atributos categóricos
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(fill_value='unknown',strategy='constant')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_features),
        ('categorical', categorical_pipeline, categorical_features),
    ],
    remainder='drop' # Elimina automáticamente todas las demás columnas no especificadas
)


In [13]:
MIN_INFO_GAIN = 0.005

In [14]:

pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DecisionTreeClassifier(
        criterion='entropy',
        min_impurity_decrease=MIN_INFO_GAIN,
        random_state=62
    ))
])